$$ \frac{\partial^2 u}{\partial x^2} - \frac{\partial u}{\partial t} = 0 $$
* $ T(x,t+k) = \lambda T(x-h, t) + (1 - 2\lambda) T(x,t) + \lambda T(x+h,t) $
* $ \lambda = \frac{k}{h^2} $
* $ T_{i,j} = \lambda T_{i-1,j-1} + (1 - 2\lambda) T_{i,j-1} + \lambda T_{i+1,j-1} $

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn
seaborn.set_theme()
%matplotlib widget

In [ ]:
h = 0.2
k = 0.01
x = np.arange(0, 100+h, h)
t = np.arange(0, 60+k, k)
n = len(x)
m = len(t)
boundaryConditions = [0, 0]
sigma = 0.1
mun = n//2
mu = x[mun]
initialConditions = 0.1*np.exp(-((x - mu)/sigma)**2)
# initialConditions = np.zeros(len(x))
# k = n//2
# initialConditions[k] = 0.1
# initialConditions[k-1:k+2] = 0.1
plt.plot(x, initialConditions)
print((n, m), mun, mu)

In [ ]:
T = np.zeros((n,m))
T[0,:] = boundaryConditions[0]
T[-1,:] = boundaryConditions[1]
T[:,0] = initialConditions
# T.round(3)
factor = k/h**2
for j in range(1,m):
    for i in range(1,n-1):
        T[i,j] = factor*T[i-1,j-1] + (1 - 2*factor)*T[i,j-1] + factor*T[i+1,j-1]

T.round(3)
T.shape

In [ ]:
# T[:,-1]

In [ ]:
R = np.linspace(1,0,m)
G = 0
B = np.linspace(0,1,m)
for j in range(0, m, 10):
    plt.plot(x, T[:,j], color=[R[j],G,B[j]], alpha=0.5)
# plt.legend(t.round(3))
plt.xlabel('distance [cm]')
plt.ylabel('Temperature [degree C]');

In [ ]:
u = mu/30
csigma = 10
def c(x, t):
    return np.exp(-((x - u*t)/csigma)**2)

In [ ]:
nrows = 4
ncols = 5
rtm = m//2
rt = t[rtm]
secstep_ratio = 0.01

In [ ]:
def plot_temperature(increase=True, contribution=False, fh_rate=1, steps=100):
    if contribution:
        increase = False

    baserow = nrows//2 if increase else nrows - 1
    offset = -baserow*steps*ncols
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18,2*nrows))
    if increase:
        title = "Variation of Temperature Increase at x=50"
    else:
        title = "Contribution of Temperature Increase at Time=%g (Flow/Heat Speed Rate=%g)" % (rt, fh_rate)
    fig.suptitle(title, fontsize=20)
    for k in range(nrows):
        for l in range(ncols):
            ax = axes[k,l]
            tm = rtm + offset + (k*ncols + l)*steps
            t_ = t[tm]
            ax.set_title(r"time=$%.3g_{%+.3g\;seconds}$" % (rt, secstep_ratio*(tm-rtm)))
            ax.plot(x, c(x, t_))
            # ax.set_xlim(20, 80)
            ax.set_ylim(-0.05,1.05)
            axt = ax.twinx()
            axt.grid(False)
            c_ = c(mu, t_)
            if increase:
                j = 0
                axt.plot(x, c_*T[:,j], color=[R[j],G,B[j]], alpha=0.5)
            else:
                j = int(round((tm - rtm)*fh_rate))
                if j <= 0:
                    axt.plot(x, c_*T[:,-j], color=[R[-j],G,B[-j]], alpha=0.5)
            # axt.set_xlim(20, 80)
            axt.set_ylim(-0.005, 0.105)
            ax.set_xlabel("x (cm)", fontsize=10)
            ax.set_ylabel("Concentration", fontsize=10)
            label = "Temperature Increase" if increase else "Increase Contribution"
            axt.set_ylabel(label, fontsize=8)
    fig.tight_layout()

In [ ]:
plot_temperature(increase=True)

In [ ]:
plot_temperature(contribution=True, fh_rate=0.25, steps=10)

In [ ]:
plot_temperature(contribution=True, fh_rate=1, steps=10)

In [ ]:
def temperature(i, j, i_offsets, rate=1, debug=False):
    i_offsets = np.asarray(i_offsets)
    # suppose
    # i = 25, x[i] = 5
    # j = 3000, t[j] = 30
    # 
    # c(x[i],t[j])*T[i,0]
    # + c(x[i],t[j-1])*T[i+du,1]
    # + c(x[i],t[j-2])*T[i+du*2,2]
    # + c(x[i],t[j-3])*T[i+du*3,3]
    # + ...
    
    w_ = 0
    s_ = 0
    s2_ = np.zeros(len(i_offsets))
    if debug:
        s_list = []
    flow_rate = n/m*rate
    x_ = x[i]
    for k in range(0, j+1):
        i_ = int(round(i - k*flow_rate))
        if i_ >= 0:
            w = c(x_, t[j-k])
            w_ += w
            s_ += w*T[i_,k]
            i2_ = i_ - i_offsets
            for p, q in enumerate(i2_):
                s2_[p] += w*T[q,k]
            if debug and k < 100:
                s_list.append((w, i_, T[i_,k]))

    if debug and len(s_list) > 0:
        s_array = np.array(s_list)
        # print(s_array)
        fig, ax = plt.subplots(subplot_kw=dict(projection='3d'))
        ax.plot(s_array[:,0], s_array[:,1], s_array[:,2], 'o-')
    return s_, *s2_
    
temperature(mun, 2786,  [25, 50], debug=True)

In [ ]:
# %%timeit -n 1 -r 1
i_offsets = [25, 50]  # 5/0.2, 10/0.2
y_1 = np.array([temperature(mun, j, i_offsets, rate=1) for j in range(m)])
y_h = np.array([temperature(mun, j, i_offsets, rate=0.5) for j in range(m)])
y_q = np.array([temperature(mun, j, i_offsets, rate=0.25) for j in range(m)])

In [ ]:
yc = np.array([c(mu, t_) for t_ in t])
beamx = np.ones(len(y_1))*50
beamx2 = np.ones(len(y_1))*55
beamx3 = np.ones(len(y_1))*60
fig = plt.figure(figsize=(18,4))
ax1 = fig.add_subplot(141, projection='3d')
ax2 = fig.add_subplot(142)
ax3 = fig.add_subplot(143)
ax4 = fig.add_subplot(144)
fig.suptitle("Simulated Temperature Change at UV Beam Spot (x=50)")
ax1.set_title("3D View")
ax2.set_title("2D View at X=50")
ax2.plot(t, yc, ":", label="Concentration")
ax3.set_title("2D View at X=55")
ax3.plot(t, yc, ":", label="Concentration")
ax4.set_title("2D View at X=60")
ax4.plot(t, yc, ":", label="Concentration")
axes = [ax2, ax3, ax4]
axts = []
for ax in axes:
    axt = ax.twinx()
    axt.grid(False)
    axts.append(axt)
alpha = 1
for k, (rate, y_) in enumerate([(1, y_1), (0.5, y_h), (0.25, y_q)]):
    color = 'C%d' % (k+1)
    # axts[0].plot(t, y_[:,0], alpha=alpha, color=color, label='flow:heat=%g (rate)' % rate)
    for n, axt in enumerate(axts):
        alpha_ = 1 if n == 0 else 0.5
        axt.plot(t, y_[:,n], alpha=alpha_, color=color, label='flow speed=%g' % rate)
    ax1.plot(beamx, t, y_[:,0], alpha=alpha, color=color)
    for n, bx in [(1, beamx2), (2, beamx3)]:
        ax1.plot(bx, t, y_[:,n], alpha=0.5, color=color)
axt_ylim = axts[0].get_ylim()
for axt in axts[1:]:
    axt.set_ylim(axt_ylim)
ax1.set_xlim(0, 100)
ax1.set_xlabel("x (cm)")
ax1.set_ylabel("Time (sec)")
ax1.set_zlabel("Temperature")
for ax, axt in zip(axes, axts):
    ax.set_xlabel("Time (sec)")
    ax.set_ylabel("Concentration")
    ax.legend(loc="upper left", fontsize=10)
    axt.set_ylabel("Temperature")
    axt.legend(loc="center left", fontsize=10)
fig.tight_layout()